In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [12]:
import pandas as pd
import numpy as np

import warnings

In [3]:
warnings.filterwarnings('ignore')

In [6]:
model_bagging = joblib.load('/content/drive/My Drive/Colab Notebooks/Dubai-Houses-MLOps/model_bagging_best.pkl')
model_xgb = joblib.load('/content/drive/My Drive/Colab Notebooks/Dubai-Houses-MLOps/model_xgb_best.pkl')
model_lgbm = joblib.load('/content/drive/My Drive/Colab Notebooks/Dubai-Houses-MLOps/model_lgbm_best.pkl')
model_catboost = joblib.load('/content/drive/My Drive/Colab Notebooks/Dubai-Houses-MLOps/model_catboost_best.pkl')
model_bagging = joblib.load('/content/drive/My Drive/Colab Notebooks/Dubai-Houses-MLOps/model_bagging_best.pkl')
model_stacking = joblib.load('/content/drive/My Drive/Colab Notebooks/Dubai-Houses-MLOps/model_stacking_best.pkl')

In [7]:
df_valid = pd.read_parquet('/content/drive/My Drive/Colab Notebooks/Dubai-Houses-MLOps/Data/df_valid.parquet')

In [16]:
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

In [17]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, median_absolute_error, r2_score

X_valid = df_valid.drop(['price', 'price_log1p'], axis=1)
y_valid_log = df_valid['price_log1p']
y_valid_orig = df_valid['price']  # original price scale

In [19]:
def regression_metrics(y_true_log, y_pred_log, model_name):
    # log scale metrics
    rmse_log = np.sqrt(mean_squared_error(y_true_log, y_pred_log))
    mae_log  = mean_absolute_error(y_true_log, y_pred_log)
    r2_log   = r2_score(y_true_log, y_pred_log)

    # original scale (expm1 reverses log1p)
    y_true_orig = np.expm1(y_true_log)
    y_pred_orig = np.expm1(y_pred_log)

    rmse_orig  = np.sqrt(mean_squared_error(y_true_orig, y_pred_orig))
    mae_orig   = mean_absolute_error(y_true_orig, y_pred_orig)
    medae_orig = median_absolute_error(y_true_orig, y_pred_orig)
    r2_orig    = r2_score(y_true_orig, y_pred_orig)

    # MAPE only on original scale
    mask = y_true_orig != 0
    mape = np.mean(np.abs((y_true_orig[mask] - y_pred_orig[mask]) / y_true_orig[mask])) * 100

    return {
        'Model':          model_name,
        # log scale
        'MAE (log)':      round(mae_log,   4),
        'RMSE (log)':     round(rmse_log,  4),
        'R² (log)':       round(r2_log,    4),
        # original scale
        'MAE (AED)':      round(mae_orig,  0),
        'RMSE (AED)':     round(rmse_orig, 0),
        'MedAE (AED)':    round(medae_orig,0),
        'MAPE (%)':       round(mape,      2),
        'R² (orig)':      round(r2_orig,   4),
    }

In [20]:
# predictions
y_pred_bagging  = model_bagging.predict(X_valid)
y_pred_xgb      = model_xgb.predict(X_valid)
y_pred_lgbm     = model_lgbm.predict(X_valid)
y_pred_catboost = model_catboost.predict(X_valid)
y_pred_stacking = model_stacking.predict(X_valid)

In [21]:
# collect results
results = pd.DataFrame([
    regression_metrics(y_valid_log, y_pred_bagging,  'Bagging'),
    regression_metrics(y_valid_log, y_pred_xgb,      'XGBoost'),
    regression_metrics(y_valid_log, y_pred_lgbm,     'LightGBM'),
    regression_metrics(y_valid_log, y_pred_catboost, 'CatBoost'),
    regression_metrics(y_valid_log, y_pred_stacking, 'Stacking'),
]).set_index('Model')

In [22]:
# styled output
results.style\
    .background_gradient(cmap='RdYlGn', subset=['R² (log)', 'R² (orig)'])\
    .background_gradient(cmap='RdYlGn_r', subset=['MAE (log)', 'RMSE (log)', 'MAE (AED)', 'RMSE (AED)', 'MedAE (AED)', 'MAPE (%)'])\
    .format({
        'MAE (log)':   '{:.4f}',
        'RMSE (log)':  '{:.4f}',
        'R² (log)':    '{:.4f}',
        'MAE (AED)':   '{:,.0f}',
        'RMSE (AED)':  '{:,.0f}',
        'MedAE (AED)': '{:,.0f}',
        'MAPE (%)':    '{:.2f}%',
        'R² (orig)':   '{:.4f}',
    })

,MAE (log),RMSE (log),R² (log),MAE (AED),RMSE (AED),MedAE (AED),MAPE (%),R² (orig)
Model,,,,,,,,
Bagging,0.1808,0.2799,0.9083,"884,548","2,852,991","207,274",18.09%,0.6805
XGBoost,0.1090,0.2045,0.9511,"605,653","2,222,931","95,920",11.00%,0.8060
LightGBM,0.1139,0.2035,0.9516,"609,487","2,138,216","108,111",11.63%,0.8205
CatBoost,0.1000,0.1901,0.9577,"556,793","2,063,079","87,231",10.28%,0.8329
Stacking,0.1049,0.1978,0.9542,"580,980","2,132,213","89,276",10.69%,0.8215


In [26]:
df_valid[['price', 'price_log1p']].describe()

,price,price_log1p
count,"8,276.0000","8,276.0000"
mean,"3,816,329.8644",14.5628
std,"9,739,867.8511",0.9244
min,0.0000,12.7367
25%,"1,100,000.0000",13.9108
50%,"2,000,000.0000",14.5087
75%,"3,529,750.0000",15.0767
max,"482,500,000.0000",17.3709


In [28]:
# R² (orig) = 0.83 — explains 83% of price variance
# MAPE = 10.28% — on average off by ~10% of the real price
# MedAE = 87,231 AED — median error is only 87k on properties with median price ~2M

# median price =   2,000,000  AED
# max price    = 482,500,000  AED  ← 241x the median

# MedAE is your most trustworthy metric here — 87k AED error on a 2M median property = 4.3% typical error,
# which is actually very good for real estate.